# 🔹 Transporte de Calor e Massa: Advecção e Dispersão
## Volume II – Capítulo 6 | Estudo de Caso: Rio Macaé

**Objetivos:**
- Resolver a equação de advecção-difusão 2D analiticamente
- Estimar coeficientes de dispersão com correlações de Liu e Elder
- Visualizar a evolução de plumas de contaminantes
- Preparar para problemas inversos de calibração

**Referência:** Lugon Junior et al. (2026), Vol II, Cap. 6; Fischer et al. (1979).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.special import erfc

# Importar funções do módulo src/
import sys; sys.path.append('../src')
from advection_diffusion_1d import analytical_gaussian

## 🔸 Coeficientes de Dispersão no Rio Macaé

Parâmetros do rio (dados de campo):
- Largura $B = 42.2$ m, Profundidade $D = 0.71$ m
- Vazão $Q = 6.0$ m³/s → $U = Q/(BD) \approx 0.20$ m/s
- Declividade $S_0 = 0.0005$

In [ ]:
def coeficientes_dispersao_rio(U, B, D, S0, beta=0.011, phi=0.23, kappa=0.067):
    """
    Calcula coeficientes de dispersão longitudinal, transversal e vertical
    usando correlações de Liu, Elder e Jobson-Sayre.
    """
    g = 9.81
    U_star = np.sqrt(g * D * S0)  # Velocidade de atrito
    
    # Correlações
    EL = beta * (U**2 * B**2) / (D * U_star)  # Liu
    ET = phi * D * U_star                      # Elder
    EV = kappa * D * U_star                    # Jobson-Sayre
    
    return {'U_star': U_star, 'EL': EL, 'ET': ET, 'EV': EV}

# Parâmetros do Rio Macaé
params_macao = {
    'U': 0.20, 'B': 42.2, 'D': 0.71, 'S0': 0.0005
}

disp = coeficientes_dispersao_rio(**params_macao)
print(f"Velocidade de atrito: U* = {disp['U_star']:.3f} m/s")
print(f"Dispersão longitudinal: EL = {disp['EL']:.2f} m²/s")
print(f"Dispersão transversal: ET = {disp['ET']:.4f} m²/s")
print(f"Dispersão vertical:    EV = {disp['EV']:.4f} m²/s")
print(f"\nRazão EL/ET = {disp['EL']/disp['ET']:.0f} → dispersão fortemente anisotrópica!")

## 🔸 Solução Analítica: Lançamento Instantâneo 2D

$$C(x,y,t) = \frac{M}{4\pi t \sqrt{E_L E_T}} 
\exp\left[-\frac{(x-x_s-Ut)^2}{4E_L t} - \frac{(y-y_s)^2}{4E_T t}\right]$$

In [ ]:
def solucao_gaussiana_2d(M, U, EL, ET, t, x, y, xs=0, ys=0):
    """Solução analítica 2D para lançamento instantâneo."""
    dx = x - xs - U*t
    dy = y - ys
    coef = M / (4 * np.pi * t * np.sqrt(EL * ET))
    expo = - (dx**2 / (4*EL*t) + dy**2 / (4*ET*t))
    return coef * np.exp(expo)

# Parâmetros do lançamento
M = 35.0          # kg de traçador
xs, ys = 3000, 2  # posição do lançamento [m]
t_hours = [1, 4, 8]  # tempos de interesse [horas]

# Grade espacial para visualização
x = np.linspace(2500, 6000, 200)
y = np.linspace(0, 42.2, 100)
X, Y = np.meshgrid(x, y)

# Plot para diferentes tempos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, t_h in zip(axes, t_hours):
    t = t_h * 3600  # converter para segundos
    C = solucao_gaussiana_2d(M, disp['U'], disp['EL'], disp['ET'], t, X, Y, xs, ys)
    
    cf = ax.contourf(X, Y, C, levels=20, cmap='viridis')
    ax.plot(xs + disp['U']*t, ys, 'r*', markersize=15, label='Pico teórico')
    ax.set_xlabel('Distância longitudinal x [m]')
    ax.set_ylabel('Distância transversal y [m]')
    ax.set_title(f't = {t_h} h')
    ax.legend()
    plt.colorbar(cf, ax=ax, label='Concentração [kg/m³]')
    ax.grid(alpha=0.3)

plt.suptitle('Evolução da pluma de contaminante no Rio Macaé', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 🔸 Animação: Propagação da Pluma

In [ ]:
# Animação da propagação (executar em ambiente Jupyter com widget support)
from IPython.display import HTML

def animate_pluma(frames=50, t_max=10*3600):
    """Gera animação da propagação da pluma."""
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.linspace(2500, 8000, 300)
    y = np.linspace(0, 42.2, 150)
    X, Y = np.meshgrid(x, y)
    
    def update(frame):
        ax.clear()
        t = frame * t_max / frames
        if t < 100: t = 100  # evitar divisão por zero
        
        C = solucao_gaussiana_2d(M, disp['U'], disp['EL'], disp['ET'], t, X, Y, xs, ys)
        cf = ax.contourf(X, Y, C, levels=25, cmap='plasma')
        ax.plot(xs + disp['U']*t, ys, 'w*', markersize=12)
        ax.set_xlim(2500, 8000)
        ax.set_ylim(0, 42.2)
        ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
        ax.set_title(f't = {t/3600:.1f} h | C_max = {C.max():.4f} kg/m³')
        ax.grid(alpha=0.3)
        return cf.collections
    
    anim = animation.FuncAnimation(fig, update, frames=frames, interval=100, blit=False)
    plt.close()  # evitar display duplicado
    return anim

# Executar animação (descomente para visualizar)
# anim = animate_pluma()
# HTML(anim.to_jshtml())

## 🔸 Preparação para Problema Inverso

### Função custo para calibração de $E_L$ e $E_T$

In [ ]:
def funcao_custo(params, dados_obs, U, M, xs, ys, t_obs):
    """
    Calcula soma dos quadrados dos resíduos para calibração de EL, ET.
    
    Parâmetros:
    -----------
    params : array-like
        [EL, ET] a serem estimados
    dados_obs : list of dicts
        Lista com observações: {'x':..., 'y':..., 'C':...}
    U, M, xs, ys, t_obs : parâmetros conhecidos do lançamento
    
    Retorna:
    --------
    S : float
        Soma dos quadrados dos resíduos
    """
    EL, ET = params
    if EL <= 0 or ET <= 0:
        return 1e10  # penalizar parâmetros não-físicos
    
    S = 0
    for obs in dados_obs:
        C_calc = solucao_gaussiana_2d(M, U, EL, ET, t_obs, 
                                      obs['x'], obs['y'], xs, ys)
        S += (C_calc - obs['C'])**2
    return S

# Exemplo: gerar dados sintéticos com ruído
np.random.seed(42)
EL_true, ET_true = disp['EL'], disp['ET']
t_obs = 4 * 3600  # 4 horas

# Pontos de observação simulados
pontos_obs = [
    {'x': 4500, 'y': 2, 'C': None},
    {'x': 5000, 'y': 10, 'C': None},
    {'x': 5500, 'y': 20, 'C': None},
]

for p in pontos_obs:
    C_true = solucao_gaussiana_2d(M, disp['U'], EL_true, ET_true, 
                                  t_obs, p['x'], p['y'], xs, ys)
    p['C'] = C_true * (1 + 0.02*np.random.randn())  # adicionar ruído de 2%
    print(f"Ponto ({p['x']}, {p['y']}): C_obs = {p['C']:.5f} kg/m³")

# Testar função custo com parâmetros verdadeiros
S_true = funcao_custo([EL_true, ET_true], pontos_obs, disp['U'], M, xs, ys, t_obs)
print(f"\nFunção custo com parâmetros verdadeiros: S = {S_true:.3e}")

## ✅ Exercícios Propostos

1. **[Graduação]** Calcule o comprimento de mistura transversal $L_T$ para o Rio Macaé com lançamento na margem ($K=0.28$).
2. **[Pós-Graduação]** Implemente o método de Levenberg-Marquardt para calibrar $E_L$ e $E_T$ a partir dos dados sintéticos acima.

> 💡 Dica: Use `scipy.optimize.least_squares` com a função `funcao_custo` como referência.